# 13 — Multi-Armed Bandit

## Learning Objectives
1. Understand the exploration-exploitation tradeoff in the bandit setting
2. Implement epsilon-greedy, UCB1, and Thompson Sampling and compare their regret
3. Model A/B testing as a bandit problem and compare traffic allocation strategies
4. Implement contextual LinUCB and understand how features change arm selection


In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import time

np.random.seed(42)
print("NumPy:", np.__version__)
print("All environments implemented from scratch — no gym required")


## Level 1: 10-Armed Bandit — epsilon-greedy Variants

The simplest formulation: K arms with unknown reward distributions.
At each step, the agent selects arm a_t and receives r_t ~ N(mu_a, 1).

Goal: maximise total reward E[sum_t r_t].
Regret = sum_t (mu_best - mu_{a_t}).

We compare epsilon=0 (pure greedy), epsilon=0.1, epsilon=0.5 over 1000 steps.


In [ ]:
# --- Level 1: Stationary bandit, epsilon-greedy comparison ---

class KArmedBandit:
    """K-armed bandit. True means drawn from N(0,1), rewards from N(mu_a,1)."""
    def __init__(self, k=10, seed=42):
        rng = np.random.default_rng(seed)
        self.k = k
        self.true_means = rng.standard_normal(k)
        self.best_arm = int(np.argmax(self.true_means))
        self.best_mean = float(self.true_means[self.best_arm])

    def pull(self, arm):
        return float(np.random.normal(self.true_means[arm], 1.0))


def run_epsilon_greedy(bandit, epsilon, n_steps=1000, seed=42):
    """Run epsilon-greedy agent. Returns (rewards, optimal_action_pct)."""
    np.random.seed(seed)
    Q = np.zeros(bandit.k); N = np.zeros(bandit.k)
    rewards = []; optimal_count = []
    for _ in range(n_steps):
        a = np.random.randint(bandit.k) if np.random.random() < epsilon else np.argmax(Q)
        r = bandit.pull(a)
        N[a] += 1; Q[a] += (r - Q[a]) / N[a]
        rewards.append(r)
        optimal_count.append(int(a == bandit.best_arm))
    return np.array(rewards), np.array(optimal_count)


bandit = KArmedBandit(k=10, seed=0)
print(f"True arm means: {bandit.true_means.round(2)}")
print(f"Best arm: {bandit.best_arm} (mean={bandit.best_mean:.2f})
")

n_steps = 1000
eps_values = [0.0, 0.1, 0.5]
colors = ["firebrick", "darkgreen", "steelblue"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for eps, col in zip(eps_values, colors):
    # Average over 50 runs for stable estimate
    all_rewards = []; all_opt = []
    for run in range(50):
        r, opt = run_epsilon_greedy(bandit, eps, n_steps, seed=run)
        all_rewards.append(r); all_opt.append(opt)
    mean_r = np.mean(all_rewards, axis=0)
    mean_opt = np.mean(all_opt, axis=0)

    window = 50
    smooth_r = np.convolve(mean_r, np.ones(window)/window, 'valid')
    smooth_opt = np.convolve(mean_opt, np.ones(window)/window, 'valid')

    axes[0].plot(smooth_r, label=f"eps={eps}", color=col)
    axes[1].plot(smooth_opt, label=f"eps={eps}", color=col)
    print(f"eps={eps}: avg reward={mean_r[-100:].mean():.3f} | optimal%={mean_opt[-100:].mean()*100:.1f}%")

for ax, title in zip(axes, ["Average Reward", "Optimal Action %"]):
    ax.set_xlabel("Steps"); ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle("10-Armed Bandit: epsilon-greedy (50 runs averaged)", fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/bandit_eps.png", dpi=80)
plt.show()


## Level 2: UCB1 vs Thompson Sampling on Non-Stationary Bandit

UCB1 uses an optimistic upper confidence bound to guide exploration:
a_t = argmax_a [Q(a) + c * sqrt(ln(t) / N(a))]

Thompson Sampling maintains a Beta(alpha_a, beta_a) posterior per arm
and samples from it — arms with uncertain posteriors are explored naturally.

On non-stationary bandits (means drift over time), UCB1 is slow to forget;
Thompson Sampling adapts faster because posteriors widen as counts diverge.


In [ ]:
# --- Level 2: UCB1 vs Thompson Sampling on non-stationary bandit ---

class NonStationaryBandit:
    """Arm means do a random walk each step. Best arm changes over time."""
    def __init__(self, k=10, drift=0.02, seed=42):
        self.k = k; self.drift = drift
        rng = np.random.default_rng(seed)
        self.means = rng.standard_normal(k)

    def pull(self, arm):
        r = float(np.random.normal(self.means[arm], 1.0))
        self.means += np.random.normal(0, self.drift, self.k)
        return r

    def best_arm(self): return int(np.argmax(self.means))
    def optimal_reward(self): return float(self.means[self.best_arm()])


class UCB1Agent:
    def __init__(self, k, c=2.0):
        self.k = k; self.c = c
        self.Q = np.zeros(k); self.N = np.zeros(k); self.t = 0

    def select(self):
        self.t += 1
        if self.t <= self.k: return self.t - 1
        return int(np.argmax(self.Q + self.c * np.sqrt(np.log(self.t) / (self.N + 1e-8))))

    def update(self, arm, r):
        self.N[arm] += 1; self.Q[arm] += (r - self.Q[arm]) / self.N[arm]


class ThompsonSamplingAgent:
    """Beta posterior Thompson Sampling (binary rewards via thresholding)."""
    def __init__(self, k):
        self.k = k; self.alpha = np.ones(k); self.beta = np.ones(k)

    def select(self): return int(np.argmax(np.random.beta(self.alpha, self.beta)))

    def update(self, arm, r):
        if r > 0: self.alpha[arm] += 1
        else: self.beta[arm] += 1


def run_agent(bandit_factory, agent_factory, n_steps=2000, n_runs=30):
    """Run multiple runs, return mean cumulative regret array."""
    all_regrets = []
    for run in range(n_runs):
        np.random.seed(run)
        bandit = bandit_factory(seed=run)
        agent = agent_factory()
        regret = []
        for _ in range(n_steps):
            a = agent.select()
            opt_r = bandit.optimal_reward()
            r = bandit.pull(a)
            agent.update(a, r)
            regret.append(opt_r - r)
        all_regrets.append(np.cumsum(regret))
    return np.mean(all_regrets, axis=0), np.std(all_regrets, axis=0)


print("Comparing UCB1 vs Thompson Sampling (stationary + non-stationary)...")
n_steps = 2000; k = 10

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, drift, title in zip(axes, [0.0, 0.03], ["Stationary Bandit", "Non-Stationary Bandit (drift=0.03)"]):
    def make_bandit(seed): return KArmedBandit(k=k, seed=seed) if drift==0 else NonStationaryBandit(k=k, drift=drift, seed=seed)
    for name, factory, col in [
        ("eps=0.1",  lambda: UCB1Agent(k, c=0), "gray"),
        ("UCB1",     lambda: UCB1Agent(k, c=2.0), "steelblue"),
        ("Thompson", lambda: ThompsonSamplingAgent(k), "darkgreen"),
    ]:
        mu, std = run_agent(make_bandit, factory, n_steps, n_runs=20)
        ax.plot(mu, label=name, color=col)
        ax.fill_between(range(n_steps), mu-std, mu+std, alpha=0.15, color=col)
    ax.set_xlabel("Steps"); ax.set_ylabel("Cumulative Regret")
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle("UCB1 vs Thompson Sampling: Regret Curves", fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/bandit_ucb_ts.png", dpi=80)
plt.show()
print("Note: Thompson Sampling adapts faster on non-stationary due to posterior uncertainty")


## Real-World Example 1: A/B Testing as a Multi-Armed Bandit

Traditional A/B testing allocates equal traffic to all variants until the test
period ends, then selects the winner. This wastes traffic on inferior variants.

Bandit-based A/B testing (especially Thompson Sampling) adaptively allocates more
traffic to the better-performing variant as evidence accumulates.


In [ ]:
# === A/B Testing as a Bandit ===

def ab_test_equal(true_cvr, n_visitors=1000, n_burn_in=300):
    """
    Traditional A/B: equal split during test, choose winner at end.
    Returns (conversions per variant, selected winner).
    """
    k = len(true_cvr)
    arm_counts = np.zeros(k, dtype=int)
    conversions = np.zeros(k, dtype=int)
    for v in range(n_visitors):
        arm = v % k  # round-robin allocation
        r = int(np.random.random() < true_cvr[arm])
        arm_counts[arm] += 1; conversions[arm] += r
    # Select arm with highest observed CVR after test
    obs_cvr = conversions / (arm_counts + 1)
    return arm_counts, conversions, int(np.argmax(obs_cvr))


def ab_test_thompson(true_cvr, n_visitors=1000):
    """
    Bandit A/B: Thompson Sampling adapts traffic allocation.
    Returns (arm_counts, conversions, winner).
    """
    k = len(true_cvr)
    alpha = np.ones(k); beta = np.ones(k)
    arm_counts = np.zeros(k, dtype=int)
    conversions = np.zeros(k, dtype=int)
    for _ in range(n_visitors):
        arm = int(np.argmax(np.random.beta(alpha, beta)))
        r = int(np.random.random() < true_cvr[arm])
        alpha[arm] += r; beta[arm] += 1 - r
        arm_counts[arm] += 1; conversions[arm] += r
    return arm_counts, conversions, int(np.argmax(alpha / (alpha + beta)))


# Simulate with 3 variants: CVR = 5%, 8%, 12%
true_cvr = [0.05, 0.08, 0.12]
k = len(true_cvr)
n_runs = 100; n_visitors = 1000
best_arm = int(np.argmax(true_cvr))

eq_traffic = np.zeros((n_runs, k)); eq_wins = 0
ts_traffic = np.zeros((n_runs, k)); ts_wins = 0
eq_conv_total = []; ts_conv_total = []

for run in range(n_runs):
    np.random.seed(run)
    ac, conv, winner = ab_test_equal(true_cvr, n_visitors)
    eq_traffic[run] = ac / n_visitors
    if winner == best_arm: eq_wins += 1
    eq_conv_total.append(conv.sum())

    np.random.seed(run)
    ac, conv, winner = ab_test_thompson(true_cvr, n_visitors)
    ts_traffic[run] = ac / n_visitors
    if winner == best_arm: ts_wins += 1
    ts_conv_total.append(conv.sum())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(k); width = 0.35
axes[0].bar(x - width/2, eq_traffic.mean(0), width, label="Equal split", color="steelblue", alpha=0.8)
axes[0].bar(x + width/2, ts_traffic.mean(0), width, label="Thompson Sampling", color="darkgreen", alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels([f"Variant {i}
CVR={c:.0%}" for i, c in enumerate(true_cvr)])
axes[0].set_ylabel("Traffic Allocation"); axes[0].set_title("Traffic Allocation Strategy")
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis="y")

metrics = [
    ("Correct winner %", eq_wins, ts_wins),
    ("Avg conversions", int(np.mean(eq_conv_total)), int(np.mean(ts_conv_total))),
]
for ax2_i, (label, eq_v, ts_v) in enumerate(metrics):
    bar_x = np.array([0, 1])
    axes[1].bar(bar_x + ax2_i * 2.5, [eq_v, ts_v], color=["steelblue","darkgreen"], alpha=0.8)
    axes[1].set_xticks([0, 1, 2.5, 3.5])
    axes[1].set_xticklabels(["Equal
Wins%", "TS
Wins%", "Equal
Conv", "TS
Conv"])
    axes[1].set_title("Equal Split vs Thompson Sampling (100 runs)")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("/tmp/ab_bandit.png", dpi=80)
plt.show()
print(f"Correct winner identified: Equal={eq_wins}% | Thompson={ts_wins}%")
print(f"Avg conversions:           Equal={np.mean(eq_conv_total):.0f} | Thompson={np.mean(ts_conv_total):.0f}")


## Real-World Example 2: Contextual Bandit with LinUCB

In practice, bandits usually have context: a user's features, time of day, etc.
A contextual bandit chooses arm a = argmax_a [theta_a^T x + alpha * ||x||_{A_a^{-1}}]
where x is the context vector. LinUCB maintains per-arm ridge regression models.


In [ ]:
# === Contextual Bandit: LinUCB ===

class LinUCBAgent:
    """
    LinUCB: linear reward model with upper confidence bound.
    a_t = argmax_a [theta_a^T x + alpha * sqrt(x^T A_a^{-1} x)]
    """
    def __init__(self, k, d, alpha=1.0):
        self.k = k; self.d = d; self.alpha = alpha
        # Per-arm ridge regression: A_a = X^T X + I, b_a = X^T y
        self.A = [np.eye(d) for _ in range(k)]
        self.b = [np.zeros(d) for _ in range(k)]

    def select(self, ctx):
        scores = []
        for arm in range(self.k):
            A_inv = np.linalg.inv(self.A[arm])
            theta = A_inv @ self.b[arm]
            uncertainty = np.sqrt(ctx @ A_inv @ ctx)
            scores.append(theta @ ctx + self.alpha * uncertainty)
        return int(np.argmax(scores))

    def update(self, arm, ctx, reward):
        self.A[arm] += np.outer(ctx, ctx)
        self.b[arm] += reward * ctx


def run_linucb(k=5, d=6, n_steps=800, alpha=1.0, seed=42):
    """
    Contextual bandit: true reward = context @ theta_arm + noise.
    Compare LinUCB vs random allocation.
    """
    np.random.seed(seed)
    rng = np.random.default_rng(seed)
    true_thetas = rng.standard_normal((k, d))

    agent = LinUCBAgent(k, d, alpha)
    rewards_linucb = []; rewards_random = []

    for _ in range(n_steps):
        ctx = np.random.randn(d)
        # LinUCB
        arm_luc = agent.select(ctx)
        r_luc = float(true_thetas[arm_luc] @ ctx + np.random.normal(0, 0.5))
        agent.update(arm_luc, ctx, r_luc)
        rewards_linucb.append(r_luc)
        # Random baseline
        arm_rand = np.random.randint(k)
        r_rand = float(true_thetas[arm_rand] @ ctx + np.random.normal(0, 0.5))
        rewards_random.append(r_rand)

    return rewards_linucb, rewards_random


r_linucb, r_random = run_linucb(k=5, d=6, n_steps=800, alpha=1.0)
print(f"LinUCB avg reward (last 200): {np.mean(r_linucb[-200:]):.3f}")
print(f"Random avg reward (last 200): {np.mean(r_random[-200:]):.3f}")

fig, ax = plt.subplots(figsize=(10, 4))
w = 40
ax.plot(np.convolve(r_linucb, np.ones(w)/w, 'valid'), label="LinUCB", color="darkgreen")
ax.plot(np.convolve(r_random, np.ones(w)/w, 'valid'), "--", label="Random", color="gray")
ax.set_xlabel("Step"); ax.set_ylabel("Reward")
ax.set_title("Contextual Bandit: LinUCB vs Random (5 arms, 6D context)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/linucb.png", dpi=80)
plt.show()


## Real-World Example 3: Bandits in Recommendation (Cold-Start)

When a new item enters the catalogue, there is no history (cold start).
A bandit approach explores the item early while exploiting well-known items,
naturally phasing out the new item if it performs poorly.


In [ ]:
# === Recommendation Cold-Start via Bandits ===

def recommendation_simulation(n_steps=1500, cold_start_at=500, seed=42):
    """
    Simulate: 5 existing items (known CVR) + 1 new item (cold start) at step 500.
    Thompson Sampling vs fixed epsilon-greedy.
    """
    np.random.seed(seed)
    # Items 0-4: known good/bad. Item 5: new item with CVR=0.15 (actually best)
    true_cvr = [0.05, 0.08, 0.06, 0.09, 0.07, 0.15]
    k_initial = 5; k_final = 6

    ts_alpha = np.ones(k_initial); ts_beta = np.ones(k_initial)
    eq_Q = np.zeros(k_initial); eq_N = np.zeros(k_initial)
    eps = 0.1

    ts_rewards = []; eq_rewards = []
    ts_arm_counts = np.zeros(k_final, dtype=int)
    eq_arm_counts = np.zeros(k_final, dtype=int)

    for step in range(n_steps):
        # At cold_start_at, a new item arrives
        if step == cold_start_at:
            ts_alpha = np.append(ts_alpha, 1.0)   # uniform Beta prior for new item
            ts_beta  = np.append(ts_beta, 1.0)
            eq_Q = np.append(eq_Q, 0.5)            # optimistic init for exploration
            eq_N = np.append(eq_N, 0.0)

        k_current = len(ts_alpha)

        # Thompson Sampling
        ts_arm = int(np.argmax(np.random.beta(ts_alpha, ts_beta)))
        ts_r = int(np.random.random() < true_cvr[ts_arm])
        ts_alpha[ts_arm] += ts_r; ts_beta[ts_arm] += 1 - ts_r
        ts_rewards.append(ts_r); ts_arm_counts[ts_arm] += 1

        # Epsilon-greedy
        eq_arm = np.random.randint(k_current) if np.random.random() < eps else int(np.argmax(eq_Q[:k_current]))
        eq_r = int(np.random.random() < true_cvr[eq_arm])
        eq_N[eq_arm] += 1; eq_Q[eq_arm] += (eq_r - eq_Q[eq_arm]) / eq_N[eq_arm]
        eq_rewards.append(eq_r); eq_arm_counts[eq_arm] += 1

    return ts_rewards, eq_rewards, ts_arm_counts, eq_arm_counts


ts_r, eq_r, ts_ac, eq_ac = recommendation_simulation()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
w = 50
ts_smooth = np.convolve(ts_r, np.ones(w)/w, 'valid')
eq_smooth = np.convolve(eq_r, np.ones(w)/w, 'valid')
axes[0].plot(ts_smooth, label="Thompson Sampling", color="darkgreen")
axes[0].plot(eq_smooth, "--", label="eps-greedy", color="steelblue")
axes[0].axvline(500 - w//2, color="red", linestyle=":", alpha=0.7, label="New item arrives")
axes[0].set_xlabel("Step"); axes[0].set_ylabel("CTR (smoothed)")
axes[0].set_title("Cold-Start: Thompson Sampling vs eps-greedy")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

x = np.arange(6); labels = [f"Item {i}
CVR={c:.2f}" for i, c in enumerate([0.05,0.08,0.06,0.09,0.07,0.15])]
labels[5] += "\n(NEW)"
axes[1].bar(x - 0.2, ts_ac / ts_ac.sum(), 0.4, label="Thompson", color="darkgreen", alpha=0.8)
axes[1].bar(x + 0.2, eq_ac / eq_ac.sum(), 0.4, label="eps-greedy", color="steelblue", alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(labels, fontsize=7)
axes[1].set_ylabel("Traffic Share"); axes[1].set_title("Traffic Allocation by Item")
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("/tmp/reco_bandit.png", dpi=80)
plt.show()
print(f"Thompson - new item (5) traffic share: {ts_ac[5]/ts_ac.sum():.2%}")
print(f"eps-greedy - new item (5) traffic share: {eq_ac[5]/eq_ac.sum():.2%}")


## Comparison: Regret Curves Over 2000 Rounds

Cumulative regret = sum_t (mu_best - mu_{a_t}) measures how much reward
was lost by not always picking the optimal arm.


In [ ]:
# === Regret Comparison: 3 algorithms over 2000 rounds ===

def run_comparison(k=10, n_steps=2000, n_runs=30, seed=0):
    """Return mean regret curves for eps=0.1, UCB1, Thompson."""
    methods = {
        "eps=0.0  (greedy)": lambda: ("eps", 0.0),
        "eps=0.1":            lambda: ("eps", 0.1),
        "UCB1 (c=2)":         lambda: ("ucb", 2.0),
        "Thompson":           lambda: ("ts",  None),
    }
    results = {}
    for name, factory in methods.items():
        all_regrets = []
        for run in range(n_runs):
            np.random.seed(run)
            bandit = KArmedBandit(k=k, seed=run)
            Q = np.zeros(k); N = np.zeros(k)
            alpha_ts = np.ones(k); beta_ts = np.ones(k)
            method, param = factory()
            regret = []
            for t in range(n_steps):
                if method == "eps":
                    a = np.random.randint(k) if np.random.random() < param else int(np.argmax(Q))
                elif method == "ucb":
                    t_safe = max(1, t)
                    ucb = Q + param * np.sqrt(np.log(t_safe + 1) / (N + 1e-8))
                    a = int(np.argmax(ucb))
                else:  # Thompson
                    a = int(np.argmax(np.random.beta(alpha_ts, beta_ts)))
                r = bandit.pull(a)
                N[a] += 1; Q[a] += (r - Q[a]) / N[a]
                if method == "ts":
                    if r > 0: alpha_ts[a] += 1
                    else: beta_ts[a] += 1
                regret.append(bandit.best_mean - r)
            all_regrets.append(np.cumsum(regret))
        results[name] = (np.mean(all_regrets, axis=0), np.std(all_regrets, axis=0))

    return results


print("Running regret comparison (30 seeds, 2000 steps)...")
t0 = time.time()
regret_results = run_comparison(k=10, n_steps=2000, n_runs=30)
print(f"Done in {time.time()-t0:.1f}s")

fig, ax = plt.subplots(figsize=(10, 5))
colors_map = {"eps=0.0  (greedy)": "firebrick", "eps=0.1": "steelblue",
              "UCB1 (c=2)": "purple", "Thompson": "darkgreen"}
for name, (mu, std) in regret_results.items():
    col = colors_map.get(name, "gray")
    ax.plot(mu, label=name, color=col)
    ax.fill_between(range(len(mu)), mu-std, mu+std, alpha=0.15, color=col)
ax.set_xlabel("Steps"); ax.set_ylabel("Cumulative Regret")
ax.set_title("Multi-Armed Bandit: Cumulative Regret (10 arms, 30 seeds)")
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/bandit_regret.png", dpi=80)
plt.show()
print("Final regrets:", {k: f"{v[0][-1]:.1f}" for k, v in regret_results.items()})


## Key Takeaways

**Core idea:** The bandit problem captures the fundamental exploration-exploitation
tradeoff: you must try different options to find the best one, but trying suboptimal
options costs reward. Epsilon-greedy is simple but wasteful; UCB and Thompson
Sampling make principled uncertainty-driven choices.

### Variants and When to Use

| Algorithm | Exploration | Non-stationary | Context | Computational |
|-----------|-------------|----------------|---------|--------------|
| eps-greedy | Uniform random | Poor | No | O(1) |
| UCB1 | Optimistic bounds | Poor | No | O(K) |
| Thompson | Posterior sampling | Good (reset beta) | No | O(K) |
| LinUCB | Ridge regression | Moderate | Yes | O(d^2 K) |
| Neural CB | Neural features | Moderate | Yes | High |

### Common Failure Modes

- **eps too small in early exploration:** Agent over-commits to a suboptimal arm
  before exploring alternatives. Symptom: high regret in first 100-200 steps.
  Fix: start with eps=0.5, decay toward 0.05 over time.
- **UCB constant c too large:** Agent wastes too many pulls on provably bad arms.
  Fix: tune c with validation data; typical range 0.5-2.0.
- **Thompson Sampling with non-binary rewards:** Beta priors assume Bernoulli arms.
  For Gaussian rewards, use Gaussian posteriors (Normal-Inverse-Gamma priors).
- **Non-stationarity with UCB1:** UCB1 never forgets old counts; best arm changes
  but agent keeps exploiting the historical best. Fix: sliding window UCB or decay counts.

### Related Concepts

- [14-exploration-exploitation](./14-exploration-exploitation.ipynb) — deeper coverage of exploration strategies
- [11-proximal-policy-optimization](./11-proximal-policy-optimization.ipynb) — PPO ratio objective relates to bandit
- [15-reward-shaping](./15-reward-shaping.ipynb) — shaped rewards in bandit setting


## Exercises

1. **Decay epsilon:** Implement epsilon that decays as eps(t) = 1/sqrt(t).
   Does it achieve lower final regret than fixed epsilon=0.1?

2. **Sliding window UCB:** For non-stationary bandits, only count the last W=100
   observations per arm. Compare against standard UCB1 on a drifting bandit.

3. **Bayesian regret bound:** The theoretical regret of Thompson Sampling is
   O(sqrt(KT * log(T))). For K=10, T=2000, compute this bound and compare
   to your empirical regret.

4. **LinUCB alpha sweep:** Run LinUCB with alpha in {0.1, 0.5, 1.0, 2.0}.
   Plot cumulative regret. What is the trade-off?

5. **Gaussian Thompson:** Implement Thompson Sampling for Gaussian-distributed
   rewards (use a Normal prior: mu ~ N(mu0, sigma0), update via conjugate).
   Compare to Beta Thompson on the 10-armed bandit.
